In [ ]:
import  pandas      as pd
import  numpy       as np
import  mlflow.tensorflow
import  json
import  os
from    pathlib     import Path

2026-05-18 13:12:30.370922: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [ ]:
os.chdir(Path.cwd().parent.parent)  
path_root = Path.cwd()

mlflow.set_tracking_uri(f"file:{path_root / 'mlruns'}")
experiment = mlflow.set_experiment("stock-lstm-AAPL")

print(f"Project root:    {path_root}")
print(f"Tracking URI:    {mlflow.get_tracking_uri()}")
print(f"Experiment:      {experiment.name}")
print(f"Experiment ID:   {experiment.experiment_id}")

2026/05/18 13:12:36 INFO mlflow.tracking.fluent: Experiment with name 'stock-lstm-AAPL' does not exist. Creating a new experiment.


Project root:    /home/caio/projetos/tech-challenge-fase4-lstm
Tracking URI:    file:/home/caio/projetos/tech-challenge-fase4-lstm/mlruns
Experiment:      stock-lstm-AAPL
Experiment ID:   984238432311869659


In [ ]:
models_dir = path_root / "models" / "AAPL"
reports_dir = path_root / "reports" / "AAPL"

with open(models_dir / "metadata.json", "r") as f:
    metadata = json.load(f)

with open(reports_dir / "metrics.json", "r") as f:
    reference_metrics = json.load(f)

metrics_comparison = pd.read_csv(reports_dir / "metrics_comparison.csv")

predictions = pd.read_csv(reports_dir / "predictions.csv")

print(f"Metadata: {len(metadata)} chaves")
print(f"Models testados: {metadata['models_tested']}")
print(f"Best model: {metadata['best_model']}")
print(f"\nMetrics comparison:")
print(metrics_comparison)
print(f"\nPredictions head:")
print(predictions.head())

Metadata: 27 chaves
Models testados: ['tanh', 'relu']
Best model: lstm_relu

Metrics comparison:
       model lstm_activation     mae     rmse    mape  epochs_executed
0  lstm_tanh            tanh  8.0551  10.3969  4.1794               11
1  lstm_relu            relu  7.1916   9.0314  3.7665                9

Predictions head:
       y_true     y_pred
0  165.210007  162.14243
1  165.229996  162.45224
2  166.470001  162.75182
3  167.630005  163.06370
4  166.649994  163.40060


In [ ]:
def directional_accuracy(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """
    
    Bem,esta função mede % de vezes que o modelo acertou a DIREÇÃO do movimento.
    Compara variação real (t -> t+1) com variação prevista (t -> previsão_t+1).
    """
    y_true = np.asarray(y_true).flatten()
    y_pred = np.asarray(y_pred).flatten()
    
    # Direção real: subiu (+1) ou desceu (-1) entre t e t+1
    actual_direction = np.sign(np.diff(y_true))
    
    # Direção prevista: predição vs valor real anterior
    predicted_direction = np.sign(y_pred[1:] - y_true[:-1])
    
    return float(np.mean(actual_direction == predicted_direction))

print(predictions.columns.tolist())

print(f"Shape: {predictions.shape}")
print(f"Primeiras linhas:")
print(predictions.head())
print(f"\nDescribe:")
print(predictions.describe())

predictions_comparison = pd.read_csv(reports_dir / "predictions_comparison.csv")
print(f"\nPredictions comparison - colunas: {predictions_comparison.columns.tolist()}")
print(predictions_comparison.head())

['y_true', 'y_pred']
Shape: (318, 2)
Primeiras linhas:
       y_true     y_pred
0  165.210007  162.14243
1  165.229996  162.45224
2  166.470001  162.75182
3  167.630005  163.06370
4  166.649994  163.40060

Describe:
           y_true      y_pred
count  318.000000  318.000000
mean   184.590252  178.437467
std     13.998236    8.037609
min    163.759995  162.142430
25%    173.794998  172.675063
50%    182.709999  177.861850
75%    191.277496  183.404230
max    234.820007  205.206670

Predictions comparison - colunas: ['y_true', 'y_pred_tanh', 'y_pred_relu']
       y_true  y_pred_tanh  y_pred_relu
0  165.210007    159.98466    159.89122
1  165.229996    160.31897    160.28839
2  166.470001    160.64597    160.67833
3  167.630005    160.98760    161.08480
4  166.649994    161.35797    161.50995


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

def compute_all_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    y_true = np.asarray(y_true).flatten()
    y_pred = np.asarray(y_pred).flatten()
    
    return {
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mape": float(np.mean(np.abs((y_true - y_pred) / y_true)) * 100),
        "directional_accuracy": directional_accuracy(y_true, y_pred),
    }

y_true = predictions_comparison["y_true"].values
y_pred_relu = predictions_comparison["y_pred_relu"].values
y_pred_tanh = predictions_comparison["y_pred_tanh"].values

metrics_relu = compute_all_metrics(y_true, y_pred_relu)
metrics_tanh = compute_all_metrics(y_true, y_pred_tanh)


results_df = pd.DataFrame({
    "lstm_relu": metrics_relu,
    "lstm_tanh": metrics_tanh,
}).round(4)

print("Métricas calculadas")
print(results_df)

print("\nSanity Check vs metadata.json (lstm_relu)")
print(f"MAE:   metadata={metadata['metrics']['mae']:.4f} | calculado={metrics_relu['mae']:.4f}")
print(f"RMSE:  metadata={metadata['metrics']['rmse']:.4f} | calculado={metrics_relu['rmse']:.4f}")
print(f"MAPE:  metadata={metadata['metrics']['mape']:.4f} | calculado={metrics_relu['mape']:.4f}")

Métricas calculadas
                      lstm_relu  lstm_tanh
mae                      7.1916     8.0551
rmse                     9.0314    10.3969
mape                     3.7665     4.1794
directional_accuracy     0.4637     0.4795

=== Sanity Check vs metadata.json (lstm_relu) ===
MAE:   metadata=7.1916 | calculado=7.1916
RMSE:  metadata=9.0314 | calculado=9.0314
MAPE:  metadata=3.7665 | calculado=3.7665


In [ ]:
import tensorflow as tf

models_dir = path_root / "models" / "AAPL"
reports_dir = path_root / "reports" / "AAPL"

runs_config = [
    {
        "run_name": "lstm_relu",
        "model_path": models_dir / "model_relu.keras",
        "activation": "relu",
        "metrics": metrics_relu,
    },
    {
        "run_name": "lstm_tanh",
        "model_path": models_dir / "model_tanh.keras",
        "activation": "tanh",
        "metrics": metrics_tanh,
    },
]

run_ids = {}

for cfg in runs_config:
    with mlflow.start_run(run_name=cfg["run_name"]) as run:
        mlflow.set_tag("ticker", "AAPL")
        mlflow.set_tag("architecture", cfg["run_name"])
        mlflow.set_tag("data_start", metadata["start_date"])
        mlflow.set_tag("data_end", metadata["end_date"])
        mlflow.set_tag("source", "evaluated_from_existing_artifacts")
        
        mlflow.log_param("activation", cfg["activation"])
        mlflow.log_param("sequence_length", metadata["sequence_length"])
        mlflow.log_param("lstm_units", metadata["lstm_units"])
        mlflow.log_param("dropout", metadata["dropout"])
        mlflow.log_param("learning_rate", metadata["learning_rate"])
        mlflow.log_param("batch_size", metadata["batch_size"])
        mlflow.log_param("epochs_executed", metadata["epochs_executed"])
        mlflow.log_param("features", str(metadata["features"]))
        mlflow.log_param("train_samples", metadata["train_samples"])
        mlflow.log_param("test_samples", metadata["test_samples"])
        
        for metric_name, metric_value in cfg["metrics"].items():
            mlflow.log_metric(metric_name, metric_value)

        model = tf.keras.models.load_model(cfg["model_path"], compile=False)
        mlflow.tensorflow.log_model(model, artifact_path="model")

        mlflow.log_artifact(str(models_dir / "scaler.pkl"), artifact_path="preprocessing")
        mlflow.log_artifact(str(models_dir / "metadata.json"))
        mlflow.log_artifact(str(reports_dir / "predictions_comparison.csv"))
        mlflow.log_artifact(str(reports_dir / "training_loss.png"), artifact_path="plots")
        
        run_ids[cfg["run_name"]] = run.info.run_id
        print(f"✅ Run '{cfg['run_name']}' logado | run_id: {run.info.run_id}")

print(f"\nRun IDs capturados: {run_ids}")

2026/05/18 13:15:36 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.
2026/05/18 13:15:43 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/05/18 13:15:43 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/05/18 13:15:43 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


✅ Run 'lstm_relu' logado | run_id: 265490ab63a5461a8e698462315a9a5f


2026/05/18 13:15:50 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/05/18 13:15:50 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


✅ Run 'lstm_tanh' logado | run_id: 787a2d96aef4451181576a3f77b5510c

Run IDs capturados: {'lstm_relu': '265490ab63a5461a8e698462315a9a5f', 'lstm_tanh': '787a2d96aef4451181576a3f77b5510c'}


In [ ]:
from mlflow.tracking import MlflowClient

client = MlflowClient()
model_name = "stock-lstm-AAPL"

best_run_id = run_ids["lstm_relu"]
model_uri = f"runs:/{best_run_id}/model"

registered = mlflow.register_model(model_uri=model_uri, name=model_name)

print(f"✅ Modelo registrado:")
print(f"   Nome: {registered.name}")
print(f"   Versão: {registered.version}")
print(f"   Status: {registered.status}")

client.transition_model_version_stage(
    name=model_name,
    version=registered.version,
    stage="Production",
    archive_existing_versions=True,
)

print(f"\n Versão {registered.version} promovida a Production.")

✅ Modelo registrado:
   Nome: stock-lstm-AAPL
   Versão: 1
   Status: READY

 Versão 1 promovida a Production.


Successfully registered model 'stock-lstm-AAPL'.
Created version '1' of model 'stock-lstm-AAPL'.
/tmp/ipykernel_24401/313538309.py:18: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


In [ ]:
print("Validação final")

runs_df = mlflow.search_runs(experiment_ids=[experiment.experiment_id])
print("\nRuns no experiment")
display_cols = [
    "tags.mlflow.runName",
    "metrics.rmse",
    "metrics.mae",
    "metrics.mape",
    "metrics.directional_accuracy",
]
print(runs_df[display_cols].to_string(index=False))

print(f"\n Modelo '{model_name}' no Registry")
for mv in client.search_model_versions(f"name='{model_name}'"):
    print(f"  Version {mv.version} | Stage: {mv.current_stage} | Run: {mv.run_id[:8]}...")


print(f"\nResumo")
print(f"Total de runs: {len(runs_df)}")
print(f"Tracking URI:  {mlflow.get_tracking_uri()}")
print(f"Próximo passo: subir o MLflow UI pra validação visual")

Validação final

=== Runs no experiment ===
tags.mlflow.runName  metrics.rmse  metrics.mae  metrics.mape  metrics.directional_accuracy
          lstm_tanh     10.396894     8.055087      4.179374                      0.479495
          lstm_relu      9.031410     7.191641      3.766550                      0.463722

 Modelo 'stock-lstm-AAPL' no Registry
  Version 1 | Stage: Production | Run: 265490ab...

Resumo
Total de runs: 2
Tracking URI:  file:/home/caio/projetos/tech-challenge-fase4-lstm/mlruns
Próximo passo: subir o MLflow UI pra validação visual
